In [18]:
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import KMeans
import numpy as np




In [19]:
sentence_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')


# read txt
with open('rule.txt', 'r') as file:
    rule_text = file.read()

# Split the text into lines and remove empty lines
rule_lines = [line.strip() for line in rule_text.split('\n') if line.strip()]

# Compute embeddings for each line
rule_embeddings = sentence_transformer_model.encode(rule_lines, convert_to_tensor=True)

rule_embeddings = rule_embeddings.detach().cpu().numpy()


In [20]:


def run_kmeans(embeddings, k):
    # Run KMeans
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)  # Cluster labels for each point
    
    # Find closest points to each center
    closest_points = []
    for center in kmeans.cluster_centers_:
        # Calculate distances from this center to all points
        distances = np.linalg.norm(embeddings - center, axis=1)
        # Get index of closest point
        closest_idx = np.argmin(distances)
        # Store the original text of the closest point
        closest_points.append(rule_lines[closest_idx])
    
    return labels, kmeans.cluster_centers_, closest_points

# Run clustering
labels, centers, representative_rules = run_kmeans(rule_embeddings, 10)

# Print representative rule for each cluster
print("\nRepresentative rules from each cluster:")
for i, rule in enumerate(representative_rules):
    print(f"Cluster {i}: {rule}")

# Print cluster sizes
unique_labels, counts = np.unique(labels, return_counts=True)
print("\nCluster sizes:")
for label, count in zip(unique_labels, counts):
    print(f"Cluster {label}: {count} rules")
    
    
# print rules for each cluster
for i in range(10):
    print(f"Cluster {i}:")
    for j in range(len(rule_lines)):
        if labels[j] == i:
            print(rule_lines[j])



Representative rules from each cluster:
Cluster 0: The agent must be holding nothing or have sufficient capacity to pick up an object.
Cluster 1: The 'examine' action is not admissible when not at the location of the object being examined, and the 'go to' action can be used to navigate to the location of an object making subsequent actions admissible
Cluster 2: Objects can be visible on other objects and multiple objects can be visible on the same surface
Cluster 3: The agent can put an object in a container when at the same location as the container and the object is being held.
Cluster 4: Containers must be in an open state to access or view their contents.
Cluster 5: The agent must be adjacent to an object to examine it if not the 'go to' action may be necessary to make the 'examine' action admissible
Cluster 6: The agent can pick up objects from a surface when at the same location as the object and the object is visible
Cluster 7: The agent must be at the same location as the obje

### function

In [23]:
from sentence_transformers import SentenceTransformer, util
from sklearn.cluster import KMeans
import numpy as np

def get_summary_rules(rule_lines, k, model_name='all-MiniLM-L6-v2'):
    """
    Get representative rules from a list of rules using clustering.
    
    Args:
        rule_lines (list): List of strings, each string is a rule
        k (int): Number of clusters/representative rules to return
        model_name (str): Name of the sentence transformer model to use
        
    Returns:
        dict: Dictionary containing:
            - 'representative_rules': List of k representative rules
            - 'cluster_sizes': Dictionary of cluster sizes
            - 'cluster_members': Dictionary of rules in each cluster
    """
    # Initialize the sentence transformer
    sentence_transformer_model = SentenceTransformer(model_name)
    
    # Compute embeddings for each line
    rule_embeddings = sentence_transformer_model.encode(rule_lines, convert_to_tensor=True)
    rule_embeddings = rule_embeddings.detach().cpu().numpy()
    
    # Run KMeans
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(rule_embeddings)
    
    # Find closest points to each center
    representative_rules = []
    cluster_members = {i: [] for i in range(k)}
    
    for center in kmeans.cluster_centers_:
        # Calculate distances from this center to all points
        distances = np.linalg.norm(rule_embeddings - center, axis=1)
        # Get index of closest point
        closest_idx = np.argmin(distances)
        # Store the original text of the closest point
        representative_rules.append(rule_lines[closest_idx])
    
    # Get cluster sizes and members
    unique_labels, counts = np.unique(labels, return_counts=True)
    cluster_sizes = {label: count for label, count in zip(unique_labels, counts)}
    
    # Organize rules by cluster
    for i, label in enumerate(labels):
        cluster_members[label].append(rule_lines[i])
    
    return {
        'representative_rules': representative_rules,
        'cluster_sizes': cluster_sizes,
        'cluster_members': cluster_members
    }


# read txt
with open('rule.txt', 'r') as file:
    rule_text = file.read()

# Split the text into lines and remove empty lines
rule_lines = [line.strip() for line in rule_text.split('\n') if line.strip()]

summary_rules = get_summary_rules(rule_lines, 10)['representative_rules']

for rule in summary_rules:
    print(rule)



The agent must be holding nothing or have sufficient capacity to pick up an object.
The 'examine' action is not admissible when not at the location of the object being examined, and the 'go to' action can be used to navigate to the location of an object making subsequent actions admissible
Objects can be visible on other objects and multiple objects can be visible on the same surface
The agent can put an object in a container when at the same location as the container and the object is being held.
Containers must be in an open state to access or view their contents.
The agent must be adjacent to an object to examine it if not the 'go to' action may be necessary to make the 'examine' action admissible
The agent can pick up objects from a surface when at the same location as the object and the object is visible
The agent must be at the same location as the object to view its contents or objects on it
The 'put' action is not admissible when the agent is not at the same location as the sur